# Gold Layer — Analytics Aggregations

Gold views:
- **daily_sales_fact**: Daily revenue, order count, avg order value
- **customer_metrics**: Lifetime value, total spend, days since last purchase
- **product_metrics**: Total revenue, units sold, avg price by product
- **category_trends**: 7-day moving average of revenue by category

In [1]:
%pip install pyspark==3.5.3

  Using cached pyspark-3.5.3-py2.py3-none-any.whl
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.0
    Can't uninstall 'pyspark'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_gold") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://dataops-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataops-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataops-secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark {spark.version} ready")

Spark 3.5.0 ready


In [3]:
BASE = "s3a://delta-lake/gold"
tables = ["daily_sales_fact", "customer_metrics", "product_metrics", "category_trends"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    df.printSchema()
    df.show(5, truncate=False)


DAILY_SALES_FACT: 100 rows × 13 cols
root
 |-- txn_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- completed_count: long (nullable = true)
 |-- refunded_count: long (nullable = true)
 |-- _updated_at: timestamp (nullable = true)
 |-- _load_date: string (nullable = true)

+----------+-----------+-----------+-----------+--------------+--------+------------+----------+-----------------+---------------+--------------+--------------------------+----------+
|txn_date  |customer_id|product_id |category   |store_location|quantity|gross_amount|net_amount|transaction_count|completed_count|refunded_count|_updated_at               |_load_date|
+----------+-----------+-------

In [4]:
# Top 10 customers by lifetime value
df = spark.read.format("delta").load(f"{BASE}/customer_metrics")
print("Top 10 customers by lifetime value:")
df.orderBy("customer_lifetime_value", ascending=False).select(
    "customer_id", "total_transactions", "customer_lifetime_value", "first_purchase_date", "last_purchase_date"
).show(10, truncate=False)

Top 10 customers by lifetime value:


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `lifetime_value` cannot be resolved. Did you mean one of the following? [`customer_id`, `customer_lifetime_value`, `_load_date`, `_updated_at`, `is_active`].;
'Sort ['lifetime_value DESC NULLS LAST], true
+- Relation [customer_id#4129,customer_lifetime_value#4130,total_transactions#4131L,first_purchase_date#4132,last_purchase_date#4133,avg_transaction_amount#4134,preferred_category#4135,preferred_payment_method#4136,is_active#4137,risk_score#4138,_updated_at#4139,_load_date#4140] parquet


In [ ]:
# Top products by revenue
df = spark.read.format("delta").load(f"{BASE}/product_metrics")
print("Top 10 products by revenue:")
df.orderBy("total_revenue", ascending=False).select(
    "product_id", "product_name", "category", "total_quantity_sold", "total_revenue", "last_sale_date"
).show(10, truncate=False)

In [ ]:
# Daily revenue trend
df = spark.read.format("delta").load(f"{BASE}/daily_sales_fact")
print("Daily sales trend:")
df.orderBy("txn_date").select(
    "txn_date", "gross_amount", "net_amount", "transaction_count"
).show(15, truncate=False)

In [ ]:
# Category trends with 7-day moving average
df = spark.read.format("delta").load(f"{BASE}/category_trends")
print("Category revenue trends (7-day moving avg):")
df.orderBy("category", "trend_date").show(20, truncate=False)

In [ ]:
spark.stop()